# ICM-UADE — Mapa de precios por sucursal
**INECO · Universidad Argentina de la Empresa**

Genera el mapa interactivo de precios y los rankings de cadenas para un mes dado,
descargando los datos directamente desde Google Drive.

**Pasos:**
1. Ejecutá la **Celda 1** (instala dependencias — ~1 minuto)
2. Modificá el mes en la **Celda 2** y ejecutala
3. Ejecutá la **Celda 3** y autorizá el acceso a Google Drive cuando te lo pida
4. Ejecutá todo lo demás — el notebook descarga y procesa automáticamente

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Celda 1 — Instalación (ejecutar una sola vez por sesión)
# ══════════════════════════════════════════════════════════════════
import sys, os

if not os.path.exists('/content/repo'):
    !git clone -q https://github.com/santiagoriverti/analisis_precios_minoristas_UADE.git /content/repo
    print('✓ Repositorio clonado')
else:
    !git -C /content/repo pull -q
    print('✓ Repositorio actualizado')

!pip install -q folium branca openpyxl pyarrow

sys.path.insert(0, '/content/repo/src')
print('✓ Listo para usar')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Celda 2 — CONFIGURACIÓN (única celda que necesitás modificar)
# ══════════════════════════════════════════════════════════════════

# Mes a analizar (formato MMAAAA)
MES = '042026'          # ← cambiá solo esto

# ── IDs de las carpetas de Google Drive (no modificar) ─────────────
# Fuente: INECO — SEPA histórico 2018-2023
FOLDER_ID_HISTORICO = '13GONeBs5lQCSUdBioHYk-8GhfDtIyliD'
# Fuente: INECO — SEPA 2024 a mes vencido (se actualiza cada mes)
FOLDER_ID_RECIENTE  = '1GNs9SrZ4BIoBsviBVWYYqRcsj4dwPF-I'

# Carpeta con los maestros dentro del Drive de SEPA reciente
# Dejá vacío ('') si los maestros están en la raíz de la misma carpeta
MAESTROS_SUBFOLDER = 'maestros'

# ── Derivados automáticos ───────────────────────────────────────────
anio = int(MES[2:])      # 2026
mes_num = int(MES[:2])   # 4
MES_FMT = f'{anio}-{mes_num:02d}'  # '2026-04'

# Archivos a buscar en Drive
ARCHIVO_PARTE1 = f'{MES}_pais_parte1COMPLETO.csv.gz'
ARCHIVO_PARTE2 = f'{MES}_pais_parte2COMPLETO.csv.gz'

# Carpeta local donde se descargan los datos en Colab
DIR_LOCAL = f'/content/datos_sepa/{MES}'
os.makedirs(DIR_LOCAL, exist_ok=True)

# Usar carpeta histórica o reciente según el año
FOLDER_ID = FOLDER_ID_HISTORICO if anio <= 2023 else FOLDER_ID_RECIENTE

print(f'Mes a analizar: {MES_FMT}')
print(f'Archivos a descargar: {ARCHIVO_PARTE1}, {ARCHIVO_PARTE2}')
print(f'Carpeta Drive: {FOLDER_ID}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Celda 3 — Autenticación con Google Drive
# ══════════════════════════════════════════════════════════════════
# Te va a pedir que inicies sesión con tu cuenta Google.
# Usá la misma cuenta que tiene acceso a las carpetas de Drive de INECO.
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
drive = build('drive', 'v3', cache_discovery=False)
print('✓ Autenticado correctamente')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Celda 4 — Descargar archivos SEPA desde Drive
# ══════════════════════════════════════════════════════════════════
from googleapiclient.http import MediaIoBaseDownload
from pathlib import Path
import io, time

def buscar_archivo_en_drive(folder_id, nombre_archivo):
    """Busca un archivo por nombre en una carpeta de Drive (y sus subcarpetas)."""
    # Buscar primero en la carpeta raíz
    query = f"'{folder_id}' in parents and name = '{nombre_archivo}' and trashed = false"
    resultado = drive.files().list(q=query, fields='files(id, name, size)').execute()
    archivos = resultado.get('files', [])
    if archivos:
        return archivos[0]

    # Buscar en subcarpetas si no se encontró en la raíz
    subcarpetas = drive.files().list(
        q=f"'{folder_id}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
        fields='files(id, name)'
    ).execute().get('files', [])

    for sub in subcarpetas:
        query_sub = f"'{sub['id']}' in parents and name = '{nombre_archivo}' and trashed = false"
        res_sub = drive.files().list(q=query_sub, fields='files(id, name, size)').execute()
        archivos_sub = res_sub.get('files', [])
        if archivos_sub:
            return archivos_sub[0]

    return None


def descargar_de_drive(file_id, ruta_local, nombre):
    """Descarga un archivo de Drive mostrando el progreso."""
    ruta = Path(ruta_local)
    if ruta.exists() and ruta.stat().st_size > 1_000_000:
        print(f'  ↩ {nombre} ya descargado ({ruta.stat().st_size/1e6:.0f} MB)')
        return

    request = drive.files().get_media(fileId=file_id)
    with open(ruta_local, 'wb') as f:
        downloader = MediaIoBaseDownload(f, request, chunksize=50*1024*1024)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f'  {nombre}: {int(status.progress() * 100)}%', end='\r')
    mb = Path(ruta_local).stat().st_size / 1e6
    print(f'  ✓ {nombre}: descargado ({mb:.0f} MB)          ')


def buscar_maestro(nombre_maestro):
    """Busca un maestro primero en la subcarpeta 'maestros', luego en la raíz."""
    # Buscar subcarpeta 'maestros'
    sub_query = f"'{FOLDER_ID_RECIENTE}' in parents and name = '{MAESTROS_SUBFOLDER}' and mimeType = 'application/vnd.google-apps.folder'"
    subs = drive.files().list(q=sub_query, fields='files(id)').execute().get('files', [])
    if subs:
        busqueda = buscar_archivo_en_drive(subs[0]['id'], nombre_maestro)
        if busqueda:
            return busqueda
    # Buscar en la carpeta raíz
    return buscar_archivo_en_drive(FOLDER_ID_RECIENTE, nombre_maestro)


# ── Descargar archivos del mes ───────────────────────────────────────────
print(f'Buscando archivos de {MES_FMT} en Drive...')

archivos_sepa = {
    ARCHIVO_PARTE1: f'{DIR_LOCAL}/{ARCHIVO_PARTE1}',
    ARCHIVO_PARTE2: f'{DIR_LOCAL}/{ARCHIVO_PARTE2}',
}

for nombre, ruta_local in archivos_sepa.items():
    info = buscar_archivo_en_drive(FOLDER_ID, nombre)
    if info is None:
        print(f'  ✗ No encontrado: {nombre}')
        print(f'    Verificá que el archivo exista en la carpeta de Drive.')
    else:
        mb = int(info.get('size', 0)) / 1e6
        print(f'  Encontrado: {nombre} ({mb:.0f} MB)')
        descargar_de_drive(info['id'], ruta_local, nombre)

# ── Descargar maestros ───────────────────────────────────────────────────
print('\nBuscando maestros...')
maestros = {
    'Maestro de Productos Interno.xlsx': f'{DIR_LOCAL}/Maestro de Productos Interno.xlsx',
    'maestro_sucursales_completo.xlsx':  f'{DIR_LOCAL}/maestro_sucursales_completo.xlsx',
}
for nombre, ruta_local in maestros.items():
    info = buscar_maestro(nombre)
    if info is None:
        print(f'  ✗ Maestro no encontrado: {nombre}')
        print(f'    Asegurate de tener la carpeta "maestros" dentro de la carpeta Drive de SEPA.')
    else:
        descargar_de_drive(info['id'], ruta_local, nombre)

print('\n✓ Descarga completa. Verificando...')
for nombre, ruta in {**archivos_sepa, **maestros}.items():
    existe = Path(ruta).exists()
    print(f'  {"✓" if existe else "✗"} {nombre}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Celda 5 — Cargar y limpiar datos SEPA
# ══════════════════════════════════════════════════════════════════
import gc
import pandas as pd
from pathlib import Path

from sepa.pipeline.loader import iter_semester_csvgz, load_master_products, load_master_branches
from sepa.pipeline.cleaner import clean_prices
from sepa.pipeline.enricher import enrich_with_branches, filter_excluded_chains
from sepa.config.canasta import CANASTA_EANS

print(f'Cargando precios de {MES_FMT}...')
frames = []
for chunk in iter_semester_csvgz(Path(DIR_LOCAL), ean_filter=CANASTA_EANS):
    chunk = filter_excluded_chains(chunk)
    chunk = clean_prices(chunk, auto_scale=True)
    frames.append(chunk)
    gc.collect()

if not frames:
    raise RuntimeError('Sin datos. Verificá que los CSV.GZ se descargaron correctamente.')

df_long = pd.concat(frames, ignore_index=True)
del frames; gc.collect()

print(f'✓ {len(df_long):,} registros cargados')
print(f'  EANs de canasta encontrados: {df_long["ean_norm"].nunique()} de 30')
print(f'  Período: {df_long["fecha"].min().date()} → {df_long["fecha"].max().date()}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Celda 6 — Enriquecer con maestros
# ══════════════════════════════════════════════════════════════════
mb = load_master_branches(f'{DIR_LOCAL}/maestro_sucursales_completo.xlsx')
df_enriched = enrich_with_branches(df_long, mb)

branch_cols = [c for c in ['id_comercio','id_bandera','id_sucursal'] if c in df_enriched.columns]
n_suc = len(df_enriched[branch_cols].drop_duplicates())

print(f'✓ Sucursales con datos: {n_suc:,}')
if 'provincia' in df_enriched.columns:
    print(f'  Provincias: {df_enriched["provincia"].nunique()}')
if 'cadena' in df_enriched.columns:
    print(f'  Cadenas: {df_enriched["cadena"].nunique()}')
if 'sucursales_tipo' in df_enriched.columns:
    print(f'  (sucursales Web excluidas automáticamente)')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Celda 7 — Calcular canasta mensual por sucursal
# ══════════════════════════════════════════════════════════════════
from sepa.pipeline.aggregator import compute_monthly_avg, build_branch_basket

df_monthly = compute_monthly_avg(df_enriched)
df_branch  = build_branch_basket(df_monthly)

df_mes = df_branch[df_branch['mes'] == MES_FMT].copy()
if df_mes.empty:
    meses = sorted(df_branch['mes'].unique())
    raise RuntimeError(f'Sin datos para {MES_FMT}. Meses disponibles: {meses}')

avg = df_mes['canasta_total'].mean()
med = df_mes['canasta_total'].median()
cv  = df_mes['canasta_total'].std() / avg * 100

print(f'══ ICM-UADE — {MES_FMT} ══')
print(f'Sucursales analizadas: {len(df_mes):,}')
print(f'Canasta promedio:  ${avg:>12,.0f}'.replace(',', '.'))
print(f'Canasta mediana:   ${med:>12,.0f}'.replace(',', '.'))
print(f'Dispersión (CV):     {cv:>10.1f}%')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Celda 8 — Mapa interactivo por sucursal
# ══════════════════════════════════════════════════════════════════
from sepa.viz.maps import make_branch_map
from IPython.display import IFrame

suc_info_cols = branch_cols + [c for c in ['lat','lng','cadena','provincia','region',
                                             'sucursales_nombre','localidad_nombre','barrio_caba']
                                if c in df_enriched.columns]
df_suc = df_enriched[suc_info_cols].drop_duplicates(subset=branch_cols)

mapa_path = f'/content/mapa_canasta_pais_{MES}_filtros.html'
make_branch_map(df_mes, df_suc, mapa_path, mes=MES_FMT)
print(f'✓ Mapa generado ({Path(mapa_path).stat().st_size/1e6:.0f} MB)')

# Mostrar inline
IFrame(mapa_path, width='100%', height=600)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Celda 9 — Rankings de cadenas (nacional y AMBA)
# ══════════════════════════════════════════════════════════════════
from sepa.analysis.chains import national_ranking, amba_ranking
from sepa.viz.charts import plot_chain_ranking
from IPython.display import Image, display as ipy_display

rank_nac  = national_ranking(df_mes, df_suc, mes=MES_FMT)
rank_amba = amba_ranking(df_mes, df_suc, mes=MES_FMT)

print('── Ranking Nacional ──')
display(rank_nac[['ranking','cadena','canasta_promedio','n_sucursales','vs_promedio_pct']]
        .style.format({'canasta_promedio': '${:,.0f}', 'vs_promedio_pct': '{:+.1f}%'})
        .hide(axis='index'))

png_nac = f'/content/ranking_cadenas_nacional_{MES}.png'
plot_chain_ranking(rank_nac, png_nac, title=f'Ranking Nacional — {MES_FMT}')
ipy_display(Image(png_nac))

if not rank_amba.empty:
    print('\n── Ranking AMBA ──')
    display(rank_amba[['ranking','cadena','canasta_promedio','n_sucursales','vs_promedio_pct']]
            .style.format({'canasta_promedio': '${:,.0f}', 'vs_promedio_pct': '{:+.1f}%'})
            .hide(axis='index'))
    png_amba = f'/content/ranking_cadenas_amba_{MES}.png'
    plot_chain_ranking(rank_amba, png_amba, title=f'Ranking AMBA — {MES_FMT}')
    ipy_display(Image(png_amba))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Celda 10 — Ranking por provincia
# ══════════════════════════════════════════════════════════════════
from sepa.pipeline.aggregator import aggregate_by_province
from sepa.analysis.basket import basket_by_province
from sepa.viz.charts import plot_province_ranking

if 'provincia' in df_suc.columns:
    df_prov = aggregate_by_province(df_mes, df_suc)
    df_rank_prov = basket_by_province(df_prov, MES_FMT)

    print('── Ranking Provincial ──')
    display(df_rank_prov[['ranking','provincia','canasta_provincia','vs_promedio_pct']]
            .style.format({'canasta_provincia': '${:,.0f}', 'vs_promedio_pct': '{:+.1f}%'})
            .hide(axis='index'))

    png_prov = f'/content/ranking_provincias_{MES}.png'
    plot_province_ranking(df_prov, png_prov, mes=MES_FMT)
    ipy_display(Image(png_prov))
else:
    print('Sin datos de provincia — verificar maestro de sucursales')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Celda 11 — Ranking de barrios CABA
# ══════════════════════════════════════════════════════════════════
from sepa.analysis.basket import barrio_ranking_caba

if 'barrio_caba' in df_suc.columns:
    df_barrios = barrio_ranking_caba(df_mes, df_suc, mes=MES_FMT)
    if not df_barrios.empty:
        print(f'── Ranking Barrios CABA ({len(df_barrios)} barrios) ──')
        display(df_barrios[['ranking','barrio_caba','canasta_barrio','n_sucursales','vs_promedio_caba_pct']]
                .rename(columns={'barrio_caba':'barrio','canasta_barrio':'canasta ($)','vs_promedio_caba_pct':'vs. CABA (%)'})
                .style.format({'canasta ($)': '${:,.0f}', 'vs. CABA (%)': '{:+.1f}%'})
                .hide(axis='index'))
else:
    print('Barrios CABA no disponibles (coordenadas no presentes en el maestro de sucursales)')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Celda 12 — Guardar outputs en Google Drive
# ══════════════════════════════════════════════════════════════════
import shutil
from googleapiclient.http import MediaFileUpload

# Buscar (o crear) la carpeta de outputs dentro del Drive de SEPA
def get_or_create_drive_folder(parent_id, nombre):
    query = f"'{parent_id}' in parents and name = '{nombre}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    res = drive.files().list(q=query, fields='files(id)').execute().get('files', [])
    if res:
        return res[0]['id']
    meta = {'name': nombre, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [parent_id]}
    return drive.files().create(body=meta, fields='id').execute()['id']

def subir_a_drive(ruta_local, folder_id, nombre_drive):
    if not Path(ruta_local).exists():
        print(f'  ⚠ No encontrado localmente: {ruta_local}')
        return
    # Detectar tipo MIME
    ext = Path(ruta_local).suffix
    mime_types = {'.html': 'text/html', '.png': 'image/png', '.xlsx': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'}
    mime = mime_types.get(ext, 'application/octet-stream')

    # Si el archivo ya existe en Drive, actualizarlo; sino, crearlo
    query = f"'{folder_id}' in parents and name = '{nombre_drive}' and trashed = false"
    existentes = drive.files().list(q=query, fields='files(id)').execute().get('files', [])
    media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)

    if existentes:
        drive.files().update(fileId=existentes[0]['id'], media_body=media).execute()
    else:
        meta = {'name': nombre_drive, 'parents': [folder_id]}
        drive.files().create(body=meta, media_body=media, fields='id').execute()

    mb = Path(ruta_local).stat().st_size / 1e6
    print(f'  ✓ {nombre_drive} subido a Drive ({mb:.1f} MB)')

# Crear carpeta outputs/MMAAAA en Drive
outputs_folder = get_or_create_drive_folder(FOLDER_ID_RECIENTE, 'outputs')
mes_folder     = get_or_create_drive_folder(outputs_folder, MES)

outputs = [
    (f'/content/mapa_canasta_pais_{MES}_filtros.html',     f'mapa_canasta_pais_{MES}_filtros.html'),
    (f'/content/ranking_cadenas_nacional_{MES}.png',        f'ranking_cadenas_nacional_{MES}.png'),
    (f'/content/ranking_cadenas_amba_{MES}.png',            f'ranking_cadenas_amba_{MES}.png'),
    (f'/content/ranking_provincias_{MES}.png',              f'ranking_provincias_{MES}.png'),
]

print(f'Subiendo outputs a Drive (carpeta outputs/{MES})...')
for ruta, nombre in outputs:
    subir_a_drive(ruta, mes_folder, nombre)

print(f'\n✓ Listo. Los archivos están en Drive → outputs/{MES}/')